# Notebook 5: The Seismic Entropy Index

Exploring whether Shannon entropy of magnitude distributions carries information about the approach of large events.

**Important caveat:** Earthquake prediction remains an unsolved problem. This analysis is exploratory. We expect any signal to be weak at best — the null result is a valid finding.

In [1]:
import sys
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.entropy import (shannon_entropy, rolling_entropy, detect_anomalies, null_model_test)
from src.gutenberg_richter import compute_bvalue_grid
from src.spatial import assign_cells
from src.plotting import (setup_style, save_figure, plot_entropy_timeseries, plot_superposed_epoch)
from src.external_data import (
    load_pb2002_boundaries, classify_grid_tectonic_settings,
    load_scedc_catalog
)

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_global.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
mc = estimate_mc(df["mag"].values)
print(f"Loaded {len(df):,} events, estimated global Mc = {mc}")

# M7+ events for association tests
m7plus = df[df["mag"] >= 7.0].sort_values("time")
print(f"M7+ events: {len(m7plus)}")

Loaded 681,450 events, estimated global Mc = 2.8
M7+ events: 387


## 5.1 Global Entropy Time Series

In [3]:
entropy_df = rolling_entropy(df["time"].values, df["mag"].values, mc,
                             window_days=90, stride_days=7, min_events=100)
# Make center_time tz-aware to match df
entropy_df["center_time"] = pd.to_datetime(entropy_df["center_time"], utc=True)

print(f"Entropy windows: {len(entropy_df)} ({entropy_df['H'].notna().sum()} valid)")
print(f"H range: {entropy_df['H'].min():.3f} → {entropy_df['H'].max():.3f}")

anomalies = detect_anomalies(entropy_df, percentile=5)
print(f"Entropy anomalies (5th percentile): {len(anomalies)}")

fig, ax = plt.subplots(figsize=(14, 6))
plot_entropy_timeseries(entropy_df, large_events=m7plus["time"].values,
                        anomalies=anomalies,
                        title="Global Shannon Entropy of Magnitude Distribution", ax=ax)
save_figure(fig, "05_global_entropy_timeseries")
plt.show()

Entropy windows: 1344 (1344 valid)
H range: 1.945 → 2.560
Entropy anomalies (5th percentile): 68


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_52131/293823812.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.2 Global Temporal Association Test

In [4]:
print("Running null model test (this may take a few minutes)...")
test_result = null_model_test(
    df["time"].values, df["mag"].values, mc,
    large_event_times=m7plus["time"].values,
    window_days=90, stride_days=7, min_events=100,
    association_days=180, percentile=5,
    n_shuffles=50,  # reduced for computational speed
    seed=42,
)

print(f"Observed hit rate: {test_result['observed_hit_rate']:.3f}")
print(f"Null model mean:   {np.mean(test_result['null_hit_rates']):.3f}")
print(f"Null model std:    {np.std(test_result['null_hit_rates']):.3f}")
print(f"p-value:           {test_result['p_value']:.3f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(test_result["null_hit_rates"], bins=30, color="#AAAAAA", alpha=0.7,
        edgecolor="white", label="Null model")
ax.axvline(test_result["observed_hit_rate"], color="#E63946", linewidth=2,
           linestyle="--", label=f"Observed ({test_result['observed_hit_rate']:.3f})")
ax.set_xlabel("Hit Rate")
ax.set_ylabel("Count")
ax.set_title(f"Entropy Anomaly \u2192 M7+ Association Test (p = {test_result['p_value']:.3f})")
ax.legend()
save_figure(fig, "05_null_model_test")
plt.show()

Running null model test (this may take a few minutes)...


Observed hit rate: 1.000
Null model mean:   1.000
Null model std:    0.000
p-value:           1.000


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_52131/949656807.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.3 Superposed Epoch Analysis

In [5]:
# Align entropy curves around M7+ events (±365 days)
aligned_curves = []
entropy_times = entropy_df["center_time"]  # already tz-aware from cell above
entropy_H = entropy_df["H"].values

for _, eq in m7plus.iterrows():
    eq_time = pd.Timestamp(eq["time"])
    if eq_time.tzinfo is None:
        eq_time = eq_time.tz_localize("UTC")
    # Find entropy values within ±365 days
    dt = (entropy_times - eq_time).dt.total_seconds() / 86400.0
    mask = (dt >= -365) & (dt <= 365)
    
    if mask.sum() < 50:
        continue
    
    # Interpolate to regular grid
    t_rel = dt[mask].values
    h_vals = entropy_H[mask]
    
    # Regular grid from -365 to 365
    t_grid = np.linspace(-365, 365, 105)  # ~7-day resolution
    h_interp = np.interp(t_grid, t_rel, h_vals)
    aligned_curves.append(h_interp)

aligned_curves = np.array(aligned_curves)
print(f"Aligned {len(aligned_curves)} M7+ events")

# Null model: shuffle and repeat
rng = np.random.default_rng(42)
n_null = 10  # reduced for speed
null_curves = []
for ni in range(n_null):
    print(f"  Null iteration {ni+1}/{n_null}...", end="\r")
    shuffled_mags = rng.permutation(df["mag"].values)
    null_ent = rolling_entropy(df["time"].values, shuffled_mags, mc,
                               window_days=90, stride_days=7, min_events=100)
    null_ent["center_time"] = pd.to_datetime(null_ent["center_time"], utc=True)
    null_H = null_ent["H"].values
    null_times = null_ent["center_time"]
    
    null_aligned = []
    for _, eq in m7plus.iterrows():
        eq_time = pd.Timestamp(eq["time"])
        if eq_time.tzinfo is None:
            eq_time = eq_time.tz_localize("UTC")
        dt = (null_times - eq_time).dt.total_seconds() / 86400.0
        mask_n = (dt >= -365) & (dt <= 365)
        if mask_n.sum() < 50:
            continue
        t_grid = np.linspace(-365, 365, 105)
        h_interp = np.interp(t_grid, dt[mask_n].values, null_H[mask_n])
        null_aligned.append(h_interp)
    
    if null_aligned:
        null_curves.append(np.nanmean(null_aligned, axis=0))

print(f"Completed {n_null} null iterations      ")
null_curves = np.array(null_curves)
null_mean = np.nanmean(null_curves, axis=0) if len(null_curves) > 0 else None
null_ci = (np.nanpercentile(null_curves, 2.5, axis=0),
           np.nanpercentile(null_curves, 97.5, axis=0)) if len(null_curves) > 0 else None

fig, ax = plt.subplots(figsize=(12, 7))
plot_superposed_epoch(aligned_curves, null_mean=null_mean, null_ci=null_ci, ax=ax)
save_figure(fig, "05_superposed_epoch")
plt.show()

Aligned 384 M7+ events


Completed 10 null iterations      


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_52131/3011951449.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.4 Regional Entropy Analysis

In [6]:
regions = {
    "Japan": {"lat": (25, 50), "lon": (125, 155)},
    "California": {"lat": (32, 42), "lon": (-125, -114)},
    "Indonesia-Philippines": {"lat": (-10, 20), "lon": (95, 140)},
    "Mediterranean": {"lat": (30, 45), "lon": (-10, 40)},
    "South America": {"lat": (-45, 5), "lon": (-80, -65)},
}

# Load SCEDC catalog for enhanced California analysis
try:
    scedc = load_scedc_catalog()
    scedc_loaded = True
    print(f"SCEDC catalog loaded: {len(scedc):,} events")
except FileNotFoundError:
    scedc_loaded = False
    print("SCEDC catalog not found, using USGS only for California")

fig, axes = plt.subplots(5, 1, figsize=(14, 20), sharex=True)

regional_results = {}

for i, (region_name, bounds) in enumerate(regions.items()):
    ax = axes[i]
    
    # For California, use SCEDC catalog (4× more events) if available
    if region_name == "California" and scedc_loaded:
        rdf = scedc.copy()
        rdf = rdf.rename(columns={"mag": "mag"})  # already has mag column
        source_label = "SCEDC"
    else:
        mask = ((df["latitude"] >= bounds["lat"][0]) & (df["latitude"] <= bounds["lat"][1]) &
                (df["longitude"] >= bounds["lon"][0]) & (df["longitude"] <= bounds["lon"][1]))
        rdf = df[mask].copy()
        source_label = "USGS"
    
    if len(rdf) < 500:
        ax.set_title(f"{region_name} — insufficient data ({len(rdf)} events)")
        continue
    
    r_mc = estimate_mc(rdf["mag"].values)
    r_entropy = rolling_entropy(rdf["time"].values, rdf["mag"].values, r_mc,
                                window_days=90, stride_days=7, min_events=100)
    r_entropy["center_time"] = pd.to_datetime(r_entropy["center_time"], utc=True)
    
    # Regional M6.5+ events (always from USGS for consistency)
    usgs_mask = ((df["latitude"] >= bounds["lat"][0]) & (df["latitude"] <= bounds["lat"][1]) &
                 (df["longitude"] >= bounds["lon"][0]) & (df["longitude"] <= bounds["lon"][1]))
    r_large = df[usgs_mask & (df["mag"] >= 6.5)]
    
    r_anomalies = detect_anomalies(r_entropy, percentile=5)
    
    plot_entropy_timeseries(r_entropy, large_events=r_large["time"].values,
                           anomalies=r_anomalies,
                           title=f"{region_name} [{source_label}] (Mc={r_mc:.1f}, n={len(rdf):,})",
                           ax=ax)
    
    regional_results[region_name] = {
        "entropy": r_entropy,
        "mc": r_mc,
        "n_events": len(rdf),
        "n_large": len(r_large),
        "source": source_label,
    }
    print(f"{region_name} [{source_label}]: {len(rdf):,} events, Mc={r_mc}, "
          f"M6.5+: {len(r_large)}, entropy windows: {len(r_entropy)}")

fig.tight_layout()
save_figure(fig, "05_regional_entropy_panels")
plt.show()

SCEDC catalog loaded: 151,966 events


Japan [USGS]: 37,075 events, Mc=4.6, M6.5+: 109, entropy windows: 1344


California [SCEDC]: 151,966 events, Mc=1.2, M6.5+: 5, entropy windows: 1292


Indonesia-Philippines [USGS]: 61,609 events, Mc=4.6, M6.5+: 184, entropy windows: 1344


Mediterranean [USGS]: 43,800 events, Mc=3.0, M6.5+: 20, entropy windows: 1344


South America [USGS]: 42,980 events, Mc=4.5, M6.5+: 100, entropy windows: 1344


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_52131/3638605660.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.5 Entropy vs. b-value

In [7]:
# Compute mean entropy per 2x2 cell
df_cells = assign_cells(df, cell_size=2.0)

cell_entropy = []
for (clat, clon), grp in df_cells.groupby(["cell_lat", "cell_lon"]):
    if len(grp) < 100:
        continue
    cell_mc = estimate_mc(grp["mag"].values)
    H = shannon_entropy(grp["mag"].values, cell_mc)
    if np.isfinite(H):
        cell_entropy.append({"cell_lat": clat, "cell_lon": clon, "mean_H": H})

entropy_cells = pd.DataFrame(cell_entropy)
bvalue_grid = compute_bvalue_grid(df, cell_size=2.0, min_events=50)

merged = entropy_cells.merge(bvalue_grid[["cell_lat", "cell_lon", "b_value"]],
                              on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(merged["b_value"], merged["mean_H"], s=10, alpha=0.4, color="#2A9D8F")
ax.set_xlabel("b-value")
ax.set_ylabel("Shannon Entropy H (bits)")
ax.set_title("Entropy vs. b-value Across 2\u00b0\u00d72\u00b0 Cells")

from scipy import stats as sp_stats
slope, intercept, r, p, se = sp_stats.linregress(merged["b_value"], merged["mean_H"])
x_fit = np.linspace(merged["b_value"].min(), merged["b_value"].max(), 50)
ax.plot(x_fit, slope * x_fit + intercept, "r--", linewidth=1.5,
        label=f"Linear fit (R\u00b2={r**2:.3f}, p={p:.1e})")
ax.legend()
save_figure(fig, "05_entropy_vs_bvalue")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_52131/502762333.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.6 Entropy by Tectonic Setting (PB2002)

Using the PB2002 plate boundary model, we classify each 2°×2° cell by tectonic setting and compare mean Shannon entropy across settings. This tests whether the information content of the magnitude distribution varies systematically with tectonic regime.

In [8]:
# Classify entropy cells by tectonic setting
boundaries = load_pb2002_boundaries()
tec_settings = classify_grid_tectonic_settings(
    entropy_cells["cell_lat"].values, entropy_cells["cell_lon"].values,
    boundaries, radius_deg=2.0
)
entropy_cells["tectonic_setting"] = tec_settings

# Also add to merged (entropy + b-value)
merged["tectonic_setting"] = classify_grid_tectonic_settings(
    merged["cell_lat"].values, merged["cell_lon"].values,
    boundaries, radius_deg=2.0
)

setting_order = ["subduction", "convergent", "transform", "spreading", "rift", "intraplate"]
setting_colors = {
    "subduction": "#E63946", "convergent": "#F4A261", "transform": "#457B9D",
    "spreading": "#2A9D8F", "rift": "#A8DADC", "intraplate": "#AAAAAA"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: Entropy by tectonic setting (box plot)
ax = axes[0]
plot_data = []
plot_labels = []
box_colors = []
for s in setting_order:
    vals = entropy_cells[entropy_cells["tectonic_setting"] == s]["mean_H"]
    if len(vals) >= 5:
        plot_data.append(vals.values)
        plot_labels.append(f"{s}\n(n={len(vals)})")
        box_colors.append(setting_colors[s])

bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("Shannon Entropy H (bits)")
ax.set_title("Magnitude Entropy by Tectonic Setting")
ax.tick_params(axis="x", rotation=30)

# Panel 2: Entropy vs b-value colored by setting
ax = axes[1]
for s in setting_order:
    mask = merged["tectonic_setting"] == s
    if mask.sum() > 0:
        ax.scatter(merged.loc[mask, "b_value"], merged.loc[mask, "mean_H"],
                   c=setting_colors[s], s=12, alpha=0.5, label=s)
ax.set_xlabel("b-value")
ax.set_ylabel("Shannon Entropy H (bits)")
ax.set_title("Entropy vs. b-value by Tectonic Setting")
ax.legend(fontsize=7, loc="lower right")

# Panel 3: Map of entropy colored by setting
ax = axes[2]
for s in setting_order:
    mask = entropy_cells["tectonic_setting"] == s
    if mask.sum() > 0:
        sc = ax.scatter(entropy_cells.loc[mask, "cell_lon"],
                        entropy_cells.loc[mask, "cell_lat"],
                        c=entropy_cells.loc[mask, "mean_H"],
                        cmap="YlOrRd", s=8, alpha=0.6,
                        vmin=1.5, vmax=3.0,
                        marker="o" if s != "intraplate" else "s")
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Global Entropy Map")
plt.colorbar(sc, ax=ax, label="H (bits)", shrink=0.8)

fig.suptitle("Shannon Entropy of Magnitude Distributions by Tectonic Setting (PB2002)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save_figure(fig, "05_entropy_by_tectonic_setting")
plt.show()

# Summary statistics
from scipy.stats import kruskal
print("Median entropy by tectonic setting:")
for s in setting_order:
    vals = entropy_cells[entropy_cells["tectonic_setting"] == s]["mean_H"]
    if len(vals) >= 5:
        print(f"  {s:15s}: median H={vals.median():.3f}, mean={vals.mean():.3f}, "
              f"std={vals.std():.3f}, n={len(vals)}")

groups = [entropy_cells[entropy_cells["tectonic_setting"] == s]["mean_H"].values
          for s in setting_order
          if len(entropy_cells[entropy_cells["tectonic_setting"] == s]) >= 5]
if len(groups) >= 2:
    H_stat, p = kruskal(*groups)
    print(f"\nKruskal-Wallis on entropy: H = {H_stat:.1f}, p = {p:.2e}")

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_52131/1159711309.py:35: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,


Median entropy by tectonic setting:
  subduction     : median H=1.113, mean=1.176, std=0.356, n=323
  convergent     : median H=1.129, mean=1.173, std=0.318, n=75
  transform      : median H=1.107, mean=1.181, std=0.352, n=103
  spreading      : median H=0.986, mean=1.013, std=0.341, n=87
  rift           : median H=1.085, mean=1.091, std=0.337, n=52
  intraplate     : median H=1.231, mean=1.246, std=0.365, n=187

Kruskal-Wallis on entropy: H = 31.0, p = 9.31e-06


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_52131/1159711309.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

Key findings from the entropy analysis:

1. **Global entropy time series** shows temporal variation in the Shannon entropy of the magnitude distribution, with identifiable low-entropy anomalies.

2. **Temporal association test** quantifies whether low-entropy anomalies occur more frequently before M7+ events than expected by chance. The p-value from the null model test indicates the statistical significance of any association.

3. **Superposed epoch analysis** stacks entropy curves aligned on M7+ events to reveal any systematic pattern. Comparison with the null model confidence interval shows whether observed patterns exceed random fluctuations.

4. **Regional analysis** reveals that entropy behavior varies across tectonic settings, reflecting differences in catalog completeness, seismicity rates, and tectonic complexity.

5. **Entropy vs. b-value** explores the relationship between two complementary measures of the magnitude distribution. A positive correlation is expected since both are sensitive to the relative proportion of small vs. large events.

**Bottom line:** Earthquake prediction remains an open problem. While entropy provides a useful summary statistic of the magnitude distribution, any apparent precursory signals must be evaluated rigorously against null models. The absence of a significant signal is itself a meaningful result.